In [1]:

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Trained adapter path
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v2" 

print("Loading model...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

Loading model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.52it/s]


Tokenizer vocab size: 128256


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Model loaded successfully!


In [3]:
# Test the model with the CORRECT tokenizer (the one with special tokens)
def test_model(patient_id, query, max_length=2000):
    # Format exactly like training data - include the explicit instruction
    messages = [
        {
            "role": "system",
            "content": """You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. The assistant gives helpful, comprehensive, and appropriate answers to the user's questions."""
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.1,              
            # top_p=0.95,                 
            # repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            
            
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

# Test with the loaded model and tokenizer
test_query = "Hi, I've been struggling with establishing and maintaining social relationships, and I'm seeking counseling to work on improving this aspect of my life. My goal is to feel more confident and connected to others, allowing me to form fulfilling friendships and build a support system.\n\nRecently, my attempts at initiating conversations or connecting with new people have proven challenging. For example, during a recent social gathering, I found myself isolated and unable to engage in meaningful interactions. This triggered feelings of self-doubt and intensified my existing insecurities.\n\nI experience symptoms such as low self-esteem, anxiety, and a persistent feeling of loneliness. These emotions often manifest before social situations, causing substantial distress. Additionally, I tend to overanalyze every interaction and worry excessively about being judged by others.\n\nMy childhood experiences of frequent relocation due to my parent's job significantly impact my ability to establish lasting relationships. This lifestyle prevented me from forming long-term connections, leaving me uncertain about how to effectively navigate social settings as an adult.\n\nTo cope with these challenges, I have tried joining clubs and attending social events, but it hasn't yielded the desired results. Are there specific techniques or strategies that could assist me in feeling more comfortable and building meaningful connections? How can I navigate social situations without overwhelming anxiety?"
response = test_model("UID_123", test_query)
print(f"Response: \n{response}")
print("="*80)

Response: 
Building and maintaining social relationships can indeed be challenging, but with some strategies and techniques, you can work towards feeling more confident and connected to others. Here are a few suggestions that may help:

1. Start small: Begin by gradually exposing yourself to social situations that make you slightly uncomfortable. This could involve attending smaller gatherings or engaging in activities where you feel more at ease. As you become more comfortable, you can gradually increase the level of challenge.

2. Practice active listening: When engaging in conversations, focus on truly listening to what the other person is saying rather than worrying about how you will respond. Show genuine interest and ask follow-up questions to keep the conversation flowing. This not only helps build rapport but also takes the spotlight off of yourself.

3. Challenge negative thoughts: Recognize and challenge any negative thoughts or beliefs you have about yourself or social inter